In [1]:
# 1. ClinVar의 "variant_summary.txt.gz" 파일을 다운로드합니다.
# 이 파일에는 모든 유전 변이와 그에 대한 임상적 중요도(레이블)가 포함되어 있습니다.
!wget https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz

print("✅ ClinVar 데이터 다운로드 완료!")

--2025-11-10 15:31:56--  https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.13, 130.14.250.31, 130.14.250.7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 411645387 (393M) [application/x-gzip]
Saving to: ‘variant_summary.txt.gz.1’

variant_summary.txt 100%[===================>] 392.58M  18.1MB/s    in 23s     

2025-11-10 15:32:21 (16.9 MB/s) - ‘variant_summary.txt.gz.1’ saved [411645387/411645387]

✅ ClinVar 데이터 다운로드 완료!


In [2]:
import pandas as pd

print(" ClinVar 파일 청크(Chunk) 단위로 로드 시작...")
print(" (파일 전체를 스캔하므로 몇 분 정도 소요됩니다)")

try:
    # 1. (수정) 파일 전체를 읽는 대신, 100만 줄씩 쪼개서 읽는 'Reader'를 준비합니다.
    chunksize = 1000000

    # 2. (추가) 처음부터 필요한 컬럼만 지정하여 메모리 사용량을 더 줄입니다.
    cols_to_load = ['Assembly', 'ClinicalSignificance', 'Type',
                    'Chromosome', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF']

    # 3. 필터링된 '알짜' 청크를 저장할 빈 리스트
    processed_chunks = []

    # 4. 100만 줄씩 반복 처리 시작
    with pd.read_csv('variant_summary.txt.gz',
                       delimiter='\t',
                       compression='gzip',
                       low_memory=False,
                       chunksize=chunksize,  # 100만 줄씩 쪼개기
                       usecols=cols_to_load) as reader: # 필요한 컬럼만 읽기

        for i, chunk in enumerate(reader):
            print(f"  ... {i+1}번째 청크 처리 중 ...")

            # 5. (동일) 원본 코드의 필터링 로직을 '각 청크'에 적용합니다.

            # 조건 1: GRCh38
            chunk = chunk[chunk['Assembly'] == 'GRCh38']

            # 조건 2: Pathogenic 또는 Benign
            target_labels = ["Pathogenic", "Benign"]
            chunk = chunk[chunk['ClinicalSignificance'].isin(target_labels)]

            # 조건 3: SNV
            chunk = chunk[chunk['Type'] == 'single nucleotide variant']

            # 6. (추가) 필터링된 '알짜' 데이터만 리스트에 추가합니다.
            processed_chunks.append(chunk)

    print("\n✅ 모든 청크 처리 완료. 하나의 데이터프레임으로 병합 중...")

    # 7. (추가) 리스트에 저장된 모든 '알짜' 청크를 하나로 합칩니다.
    final_data = pd.concat(processed_chunks, ignore_index=True)

    # 8. (동일) 원본 코드의 후반부 로직을 여기에 적용합니다.
    columns_to_keep = ['Chromosome', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'ClinicalSignificance']
    final_data = final_data[columns_to_keep].dropna().drop_duplicates()

    # 4. 레이블을 숫자로 변경 (Pathogenic=1, Benign=0)
    final_data['label'] = final_data['ClinicalSignificance'].map({'Pathogenic': 1, 'Benign': 0})

    print("✅ 데이터 전처리 완료!")
    print(f"총 {len(final_data)}개의 유효한 변이 데이터 확보.")
    print("\n[데이터 샘플]")
    print(final_data.head())

    # (선택 사항) 메모리 절약을 위해 원본 데이터(gz) 파일 삭제
    # import os
    # os.remove('variant_summary.txt.gz')

except Exception as e:
    print(f"🚨 에러 발생: {e}")
    print("파일을 다시 다운로드하거나, Colab 런타임을 초기화해보세요.")

 ClinVar 파일 청크(Chunk) 단위로 로드 시작...
 (파일 전체를 스캔하므로 몇 분 정도 소요됩니다)
  ... 1번째 청크 처리 중 ...
  ... 2번째 청크 처리 중 ...
  ... 3번째 청크 처리 중 ...
  ... 4번째 청크 처리 중 ...
  ... 5번째 청크 처리 중 ...
  ... 6번째 청크 처리 중 ...
  ... 7번째 청크 처리 중 ...
  ... 8번째 청크 처리 중 ...
  ... 9번째 청크 처리 중 ...

✅ 모든 청크 처리 완료. 하나의 데이터프레임으로 병합 중...
✅ 데이터 전처리 완료!
총 262869개의 유효한 변이 데이터 확보.

[데이터 샘플]
  Chromosome  PositionVCF ReferenceAlleleVCF AlternateAlleleVCF  \
0         11    126275389                  C                  T   
1          6     26093008                  G                  A   
2          6     26093215                  G                  T   
3          2     19989284                  T                  C   
4          2     19945787                  T                  C   

  ClinicalSignificance  label  
0           Pathogenic      1  
1               Benign      0  
2           Pathogenic      1  
3           Pathogenic      1  
4           Pathogenic      1  


3. 유전체 지도

In [5]:
import pyfaidx

print("3.1GB 유전체 지도(/tmp/hg38.fa) 파일의 '목차(.fai)' 생성을 시작합니다.")
print("이 작업은 몇 분 정도 소요될 수 있습니다...")

try:
    # pyfaidx.Fasta() 함수는 .fai 파일이 없으면 자동으로 생성해 줍니다.
    genome_map = pyfaidx.Fasta('/tmp/hg38.fa')

    print("\n✅ '목차' 생성 완료! (/tmp/hg38.fa.fai)")

    # 3. 두 파일이 모두 준비되었는지 다시 한번 검증합니다.
    print("\n[검증] /tmp/ 폴더에 두 파일이 모두 준비되었는지 확인합니다:")
    !ls -lh /tmp/hg38.fa*

except Exception as e:
    print(f"🚨 에러 발생: {e}")
    print("런타임을 초기화하고 [Phase 3] 코드를 처음부터 다시 실행해 보세요.")

3.1GB 유전체 지도(/tmp/hg38.fa) 파일의 '목차(.fai)' 생성을 시작합니다.
이 작업은 몇 분 정도 소요될 수 있습니다...

✅ '목차' 생성 완료! (/tmp/hg38.fa.fai)

[검증] /tmp/ 폴더에 두 파일이 모두 준비되었는지 확인합니다:
-rw-r--r-- 1 root root 3.1G Jan 16  2014 /tmp/hg38.fa
-rw-r--r-- 1 root root  19K Nov 10 15:57 /tmp/hg38.fa.fai


In [8]:
import pandas as pd
import pyfaidx
from tqdm import tqdm

# --- 1. 변수 및 지도 로드 (이전과 동일) ---
try:
    print(f"좌표 목록(final_data)에 총 {len(final_data)}개의 변이 정보가 있습니다.")
except NameError:
    print("🚨 [에러] 'final_data' 변수를 찾을 수 없습니다!")
    print("   [Phase 2] (ClinVar 데이터 전처리) 셀을 다시 실행하여 final_data를 생성해주세요.")
    # 이 셀을 중단하고 위로 돌아가야 합니다.

print("유전체 지도(/tmp/hg38.fa)를 메모리로 불러옵니다...")
try:
    genome_map = pyfaidx.Fasta('/tmp/hg38.fa')
    print("✅ 유전체 지도 로드 완료.")
except pyfaidx.FastaIndexingError:
     print("🚨 [에러] 유전체 목차(.fai) 파일을 찾을 수 없습니다!")
     print("   [Phase 3] (목차 생성) 셀을 다시 실행해주세요.")
     # 이 셀을 중단하고 위로 돌아가야 합니다.


# --- 2. 설정값 및 [신규] 카운터 초기화 ---
SEQ_LEN = 511
half_len = SEQ_LEN // 2 # 255

new_data_rows = []

# ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
# [수정된 부분 1] 건너뛴(skip) 데이터 추적을 위한 카운터 3개
skip_count_keyerror = 0   # 1. 'chrMT' 등 지도에 없는 염색체
skip_count_length = 0     # 2. 염색체 끝/처음이라 서열이 짧은 경우 (IndexError 해결)
skip_count_mismatch = 0   # 3. ClinVar 정보와 지도 정보가 불일치
# ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


print(f"\n총 {len(final_data)}개의 좌표로 실제 서열 추출을 시작합니다...")

# --- 3. 서열 추출 (수정된 로직) ---
for idx, row in tqdm(final_data.iterrows(), total=len(final_data)):
    try:
        # 1) 좌표 정보 가져오기
        chrom = 'chr' + str(row['Chromosome'])
        pos = int(row['PositionVCF'])
        ref_base = str(row['ReferenceAlleleVCF'])
        alt_base = str(row['AlternateAlleleVCF'])
        label = int(row['label'])

        # 2) 지도에서 서열 추출
        start = (pos - 1) - half_len
        end = (pos - 1) + half_len + 1
        ref_seq = genome_map[chrom][start:end].seq.upper()

        # ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
        # [수정된 부분 2] IndexError 방지용 길이 검사
        if len(ref_seq) != SEQ_LEN:
            skip_count_length += 1
            continue
        # ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

        # 3) '참조 서열'이 ClinVar 정보와 일치하는지 검증
        if ref_seq[half_len] != ref_base:
            # [수정된 부분 3] 불일치 카운트
            skip_count_mismatch += 1
            continue

        # 4) '변이 서열' 생성
        var_seq = ref_seq[:half_len] + alt_base + ref_seq[half_len+1:]

        # 5) 최종 데이터 리스트에 추가
        new_data_rows.append((ref_seq, var_seq, label))

    except (KeyError, ValueError) as e:
        # (예: 'chrMT' 같은 염색체 이름, 서열 범위 초과 등 예외 처리)
        # [수정된 부분 4] Key/Value 에러 카운트
        skip_count_keyerror += 1
        continue

print("\n--- 작업 완료 ---")
print(f"✅ 총 {len(new_data_rows)}개의 유효한 (참조 서열, 변이 서열, 레이블) 쌍 확보.")

# ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
# [수정된 부분 5] 건너뛴 데이터 최종 보고
print("\n[데이터 처리 요약 (건너뛴 항목)]")
print(f"  - Key/Value 에러 (e.g., 'chrMT' 등): {skip_count_keyerror} 개")
print(f"  - 서열 길이 부족 (염색체 시작/끝): {skip_count_length} 개")
print(f"  - 서열 불일치 (ClinVar != hg38): {skip_count_mismatch} 개")
total_skipped = skip_count_keyerror + skip_count_length + skip_count_mismatch
print(f"  -------------------------------------------")
print(f"  총 {total_skipped} 개 데이터 건너뜀.")
print(f"  ( {len(final_data)} - {total_skipped} = {len(new_data_rows)} )")
# ★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

# --- 4. 저장 (이전과 동일) ---
train_df = pd.DataFrame(new_data_rows, columns=['ref_seq', 'var_seq', 'label'])
print("\n[최종 학습 데이터셋 샘플]")
print(train_df.head())

print("\n[저장] 완성된 학습 데이터를 'snv_train_dataset.csv' 파일로 저장합니다...")
train_df.to_csv('snv_train_dataset.csv', index=False)
print("✅ 저장 완료! [Phase 3]의 모든 작업이 끝났습니다.")

좌표 목록(final_data)에 총 262869개의 변이 정보가 있습니다.
유전체 지도(/tmp/hg38.fa)를 메모리로 불러옵니다...
✅ 유전체 지도 로드 완료.

총 262869개의 좌표로 실제 서열 추출을 시작합니다...


100%|██████████| 262869/262869 [00:21<00:00, 12511.46it/s]



--- 작업 완료 ---
✅ 총 261931개의 유효한 (참조 서열, 변이 서열, 레이블) 쌍 확보.

[데이터 처리 요약 (건너뛴 항목)]
  - Key/Value 에러 (e.g., 'chrMT' 등): 935 개
  - 서열 길이 부족 (염색체 시작/끝): 2 개
  - 서열 불일치 (ClinVar != hg38): 1 개
  -------------------------------------------
  총 938 개 데이터 건너뜀.
  ( 262869 - 938 = 261931 )

[최종 학습 데이터셋 샘플]
                                             ref_seq  \
0  TCCTGTGTCACGGGAACTGCCCTGGGCCGTGGTAGTTCTCTGTCCT...   
1  AACTACTACCCCCAGAACATCACCATGAAGTGGCTGAAGGATAAGC...   
2  GGTATGTGACTGATGAGAGCCAGGAGCTGAGAAAATCTATTGGGGG...   
3  AACACCATAAAATGTTTTGGTAGTTTTCCCTTTAAAATAATCAGAA...   
4  AAAGAACAGATAAATAAATGCCTAGGAAGTACCTTTCAGAGAAAGT...   

                                             var_seq  label  
0  TCCTGTGTCACGGGAACTGCCCTGGGCCGTGGTAGTTCTCTGTCCT...      1  
1  AACTACTACCCCCAGAACATCACCATGAAGTGGCTGAAGGATAAGC...      0  
2  GGTATGTGACTGATGAGAGCCAGGAGCTGAGAAAATCTATTGGGGG...      1  
3  AACACCATAAAATGTTTTGGTAGTTTTCCCTTTAAAATAATCAGAA...      1  
4  AAAGAACAGATAAATAAATGCCTAGGAAGTACCTTTCAGAGAAAGT...      